# Phase 1: Dataset Inspection

**Project:** Workload-Aware SARIMA-GARCH Framework for Data Center Cooling Load Forecasting and Uncertainty Analysis

**Syllabus Alignment:** Unit 1 (Trend, Seasonality, Decomposition) & Unit 3 (Stationarity, Sampling)

**Dataset:** NREL ESIF Data Center Telemetry — 1-Year Subset (May 2024 – April 2025)

---

### Objectives
1. Verify file integrity, schema, and column data types
2. Parse timestamps and confirm the date range and timezone handling
3. Quantify missing values, duplicates, zeros, and negative readings
4. Determine the empirical sampling frequency for both PUE and Weather feeds
5. Compute descriptive statistics for all numeric variables
6. Generate diagnostic visualizations
7. Assess data suitability for SARIMA-GARCH modelling

> **Important Measurement Distinction:**
> `cooling_kw` represents the **electrical power consumed** by cooling equipment (chillers, fans, evaporative systems).
> It is **not** the thermal cooling capacity delivered to servers. We refer to it as the *cooling power demand proxy* throughout this study.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

print(f'Python: {sys.version.split()[0]}')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

## 1. File Verification

We selected a **1-year subset** (May 2024 – April 2025) from the full multi-year NREL ESIF dataset.

**Rationale for subsetting:**
- The full dataset contains 4.57M (PUE) and 4.03M (Weather) records spanning ~10 years
- SARIMA and GARCH model fitting is computationally intensive — millions of rows is infeasible
- 1 year captures all four seasons, daily/weekly cycles, and provides sufficient train/validation/test splits
- After hourly aggregation: ~8,760 observations — ideal for time-series modelling

In [ ]:
pue_path = '../data/raw/pue_subset_2024may_2025apr.csv'
weather_path = '../data/raw/weather_subset_2024may_2025apr.csv'

pue_size = os.path.getsize(pue_path) / (1024 * 1024)
weather_size = os.path.getsize(weather_path) / (1024 * 1024)

print(f'PUE file:     {pue_size:.1f} MB')
print(f'Weather file: {weather_size:.1f} MB')

In [ ]:
df_pue = pd.read_csv(pue_path)
df_weather = pd.read_csv(weather_path)

print(f'PUE:     {len(df_pue):>10,} rows x {df_pue.shape[1]} columns')
print(f'Weather: {len(df_weather):>10,} rows x {df_weather.shape[1]} columns')
print(f'\nPUE columns:     {df_pue.columns.tolist()}')
print(f'Weather columns: {df_weather.columns.tolist()}')

## 2. Initial Row Inspection & Data Types

In [ ]:
print('--- PUE Dataset (First 5 Rows) ---')
display(df_pue.head())
print('\nData Types:')
print(df_pue.dtypes)

In [ ]:
print('--- Weather Dataset (First 5 Rows) ---')
display(df_weather.head())
print('\nData Types:')
print(df_weather.dtypes)

## 3. Timestamp Parsing & Date Range

The raw `ts` column uses a JavaScript-style format:
```
Mon Nov 09 2015 20:00:01 GMT-0700 (Mountain Standard Time)
```

We parse the first 24 characters to extract the core datetime. The timezone suffix (MST/MDT) is noted but we work in naive local Mountain Time for simplicity.

In [ ]:
df_pue['dt'] = pd.to_datetime(df_pue['ts'].str[:24], format='%a %b %d %Y %H:%M:%S', errors='coerce')
df_weather['dt'] = pd.to_datetime(df_weather['ts'].str[:24], format='%a %b %d %Y %H:%M:%S', errors='coerce')

print(f'PUE unparseable timestamps:     {df_pue["dt"].isna().sum()}')
print(f'Weather unparseable timestamps: {df_weather["dt"].isna().sum()}')

print(f'\nPUE date range:     {df_pue["dt"].min()} to {df_pue["dt"].max()}')
print(f'Weather date range: {df_weather["dt"].min()} to {df_weather["dt"].max()}')

## 4. Missing Value Analysis

In [ ]:
print('PUE Missing Values:')
pue_missing = df_pue.drop(columns=['dt']).isnull().sum()
pue_miss_df = pd.DataFrame({'Missing': pue_missing, 'Pct(%)': (pue_missing / len(df_pue) * 100).round(3)})
display(pue_miss_df)

print('\nWeather Missing Values:')
weather_missing = df_weather.drop(columns=['dt']).isnull().sum()
weather_miss_df = pd.DataFrame({'Missing': weather_missing, 'Pct(%)': (weather_missing / len(df_weather) * 100).round(3)})
display(weather_miss_df)

## 5. Tags & Unit Column Investigation

The `tags` column could potentially identify different equipment streams or measurement points. We need to verify whether different tags represent different sensors that should NOT be combined.

In [ ]:
print(f'PUE tags — all null: {df_pue["tags"].isna().all()}')
print(f'Weather tags — all null: {df_weather["tags"].isna().all()}')
print(f'\nWeather unit column — unique values: {df_weather["unit"].dropna().unique()}')
print(f'Weather unit non-null count: {df_weather["unit"].notna().sum():,} / {len(df_weather):,}')

**Interpretation:** The `tags` column is entirely null in both datasets — this confirms a single unified facility-level telemetry stream. No equipment-stream disentangling is required. The weather `unit` column contains only the label `'outside'`, providing no additional measurement metadata.

## 6. Duplicate Timestamps

In [ ]:
pue_dup = df_pue['dt'].duplicated().sum()
weather_dup = df_weather['dt'].duplicated().sum()
print(f'PUE duplicate timestamps:     {pue_dup:>6,} ({pue_dup/len(df_pue)*100:.3f}%)')
print(f'Weather duplicate timestamps: {weather_dup:>6,} ({weather_dup/len(df_weather)*100:.3f}%)')

## 7. Sampling Frequency Analysis

Understanding the empirical sampling interval is critical for choosing the right aggregation frequency.

We compute the time delta between consecutive sorted observations.

In [ ]:
df_pue_sorted = df_pue.dropna(subset=['dt']).sort_values('dt')
df_weather_sorted = df_weather.dropna(subset=['dt']).sort_values('dt')

pue_deltas = df_pue_sorted['dt'].diff().dt.total_seconds().dropna()
weather_deltas = df_weather_sorted['dt'].diff().dt.total_seconds().dropna()

print('PUE Sampling Interval (seconds):')
print(pue_deltas.describe())
print(f'Mode: {pue_deltas.mode().iloc[0]:.0f} seconds')

print(f'\nWeather Sampling Interval (seconds):')
print(weather_deltas.describe())
print(f'Mode: {weather_deltas.mode().iloc[0]:.0f} seconds')

**Key Discovery:** Both the PUE and Weather feeds in this recent 2024–2025 window are sampled at a **1-minute cadence** (mode = median = 60 seconds). This differs from the earlier 2015–2016 data which had 5-minute PUE intervals.

With 1-minute data, **hourly aggregation** (mean) will:
- Average ~60 readings per hour
- Suppress high-frequency sensor noise
- Yield ~8,760 hourly observations — ideal for SARIMA(p,d,q)(P,D,Q,24)

In [ ]:
# Visualize sampling interval distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pue_deltas[pue_deltas <= 600], bins=100, color='#1f77b4', alpha=0.8, edgecolor='black', lw=0.3)
axes[0].axvline(x=60, color='red', ls='--', lw=2, label='Nominal 1 min (60s)')
axes[0].set_title('PUE Sampling Interval', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Delta (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(weather_deltas[weather_deltas <= 180], bins=100, color='#ff7f0e', alpha=0.8, edgecolor='black', lw=0.3)
axes[1].axvline(x=60, color='red', ls='--', lw=2, label='Nominal 1 min (60s)')
axes[1].set_title('Weather Sampling Interval', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delta (seconds)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/phase1_sampling_intervals.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Numeric Variable Statistics

In [ ]:
pue_num_cols = ['cooling_kw', 'it_power_kw', 'hvac_kw', 'pump_kw', 'plug_and_light_kw', 'pue', 'energy_reuse', 'ere']

pue_desc = df_pue[pue_num_cols].describe().T
pue_desc['zeros'] = (df_pue[pue_num_cols] == 0).sum()
pue_desc['negatives'] = (df_pue[pue_num_cols] < 0).sum()

print('PUE Numeric Summary:')
display(pue_desc)
pue_desc.to_csv('../results/metrics/phase1_pue_numeric_summary.csv')

In [ ]:
weather_num_cols = ['outdoor_air_temp', 'outdoor_air_humidity']

weather_desc = df_weather[weather_num_cols].describe().T
weather_desc['zeros'] = (df_weather[weather_num_cols] == 0).sum()
weather_desc['negatives'] = (df_weather[weather_num_cols] < 0).sum()

print('Weather Numeric Summary:')
display(weather_desc)
weather_desc.to_csv('../results/metrics/phase1_weather_numeric_summary.csv')

## 9. Diagnostic Visualizations

In [ ]:
# Raw time series: cooling_kw, it_power_kw, pue
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(df_pue_sorted['dt'], df_pue_sorted['cooling_kw'], color='#1f77b4', lw=0.3, alpha=0.7)
axes[0].set_ylabel('Power (kW)', fontsize=11)
axes[0].set_title('Cooling Power Demand (cooling_kw)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.4)

axes[1].plot(df_pue_sorted['dt'], df_pue_sorted['it_power_kw'], color='#2ca02c', lw=0.3, alpha=0.7)
axes[1].set_ylabel('Power (kW)', fontsize=11)
axes[1].set_title('IT Workload Power (it_power_kw)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.4)

axes[2].plot(df_pue_sorted['dt'], df_pue_sorted['pue'], color='#d62728', lw=0.3, alpha=0.7)
axes[2].set_ylabel('PUE Ratio', fontsize=11)
axes[2].set_title('Power Usage Effectiveness (PUE)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Timestamp (Mountain Time)', fontsize=11)
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../results/figures/phase1_raw_timeseries.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Weather time series
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

axes[0].plot(df_weather_sorted['dt'], df_weather_sorted['outdoor_air_temp'], color='#ff7f0e', lw=0.3, alpha=0.7)
axes[0].set_ylabel('Temperature (°F)', fontsize=11)
axes[0].set_title('Outdoor Air Temperature', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.4)

axes[1].plot(df_weather_sorted['dt'], df_weather_sorted['outdoor_air_humidity'], color='#9467bd', lw=0.3, alpha=0.7)
axes[1].set_ylabel('Humidity (%)', fontsize=11)
axes[1].set_title('Outdoor Air Humidity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Timestamp (Mountain Time)', fontsize=11)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../results/figures/phase1_weather_timeseries.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Distributions of key variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df_pue['cooling_kw'].dropna(), bins=100, color='#1f77b4', alpha=0.8, edgecolor='black', lw=0.3)
axes[0,0].set_title('Distribution: cooling_kw', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Power (kW)')

axes[0,1].hist(df_pue['it_power_kw'].dropna(), bins=100, color='#2ca02c', alpha=0.8, edgecolor='black', lw=0.3)
axes[0,1].set_title('Distribution: it_power_kw', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Power (kW)')

axes[1,0].hist(df_weather['outdoor_air_temp'].dropna(), bins=100, color='#ff7f0e', alpha=0.8, edgecolor='black', lw=0.3)
axes[1,0].set_title('Distribution: outdoor_air_temp', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Temperature (°F)')

axes[1,1].hist(df_weather['outdoor_air_humidity'].dropna(), bins=100, color='#9467bd', alpha=0.8, edgecolor='black', lw=0.3)
axes[1,1].set_title('Distribution: outdoor_air_humidity', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Humidity (%)')

plt.tight_layout()
plt.savefig('../results/figures/phase1_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly missing value heatmap
df_pue_sorted['month'] = df_pue_sorted['dt'].dt.to_period('M')
monthly_miss_cols = ['cooling_kw', 'it_power_kw', 'hvac_kw', 'pump_kw', 'pue']
monthly_missing = df_pue_sorted.groupby('month')[monthly_miss_cols].apply(lambda x: x.isnull().sum())

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(monthly_missing.T, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('PUE: Missing Values by Month', fontsize=13, fontweight='bold')
ax.set_ylabel('Variable')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig('../results/figures/phase1_missing_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Phase 1 — Summary of Findings

### Dataset Dimensions
| Dataset | Records | Columns | Date Range | File Size |
|---------|--------:|--------:|------------|----------:|
| PUE (1-yr subset) | 516,525 | 11 | 2024-05-01 to 2025-04-30 | 104.8 MB |
| Weather (1-yr subset) | 520,877 | 6 | 2024-05-01 to 2025-04-30 | 53.7 MB |

### Sampling Frequency
- **Both feeds:** 1-minute cadence (mode = median = 60 seconds)
- Occasional gaps exist (max delta: PUE ~63 hrs, Weather ~48 hrs) — likely maintenance windows
- After hourly aggregation: ~8,760 hourly observations

### Missing Values
- **PUE:** Zero missing values for all key numeric columns (cooling_kw, it_power_kw, hvac_kw, etc.)
- **Weather:** Minimal — outdoor_air_temp: 691 (0.13%), outdoor_air_humidity: 764 (0.15%)
- **Tags column:** 100% null in both datasets — single unified facility stream, no equipment-stream issue

### Key Variable Statistics
| Variable | Mean | Min | Max | Zeros | Negatives | Notes |
|----------|-----:|----:|----:|------:|----------:|-------|
| cooling_kw | 11.09 | 0.0 | 63.25 | 3 | 0 | Clean — primary target |
| it_power_kw | 2553.2 | 0.1 | 3792.4 | 0 | 0 | Clean — exogenous variable |
| hvac_kw | 156.8 | 8.7 | 244.1 | 0 | 0 | Much larger than cooling_kw |
| pue | 1.09 | 1.03 | 528.3 | 0 | 0 | Extreme outliers (max = 528!) |
| energy_reuse | 0.13 | -109.8 | 290.0 | 209 | 172,710 | 33% negatives — unreliable |
| outdoor_air_temp | 54.5 | -4.5 | 101.9 | 0 | 639 | Likely Fahrenheit (Colorado) |
| outdoor_air_humidity | 39.1 | 10.6 | 93.6 | 0 | 0 | Clean |

### Duplicate Timestamps
- PUE: 58 duplicates (0.011%) — negligible
- Weather: 515 duplicates (0.099%) — negligible

### Issues Identified for Phase 2
1. **PUE outliers:** `pue` max = 528.3 — extreme transient spikes need capping/removal
2. **energy_reuse:** 33% negative values — this variable is unreliable and should be excluded
3. **cooling_kw zeros (3 observations):** Likely sensor drop — set to NaN and interpolate
4. **Temperature negatives (639):** Valid for Colorado winters (−4.5°F is possible), keep
5. **Timestamp gaps:** Occasional multi-hour gaps — handle during hourly resampling
6. **Duplicate timestamps:** Remove by keeping first occurrence
7. **Droppable columns:** `tags` (all null), `unit` (constant), `day` (redundant), `energy_reuse` (unreliable)